# Notebook 1 — Prepare Data

**Run this notebook first, before anything else.**

This notebook loads the three CSV files, cleans them, builds the conflict matrix,
and prints summaries so you can verify the data looks correct.

You only need to run this once (or again if the data changes).

## Step 0 — Imports

In [1]:
import sys
import os
import pandas as pd

# Tell Python to look in the parent folder so it can find the 'utils' scripts
sys.path.append(os.path.abspath('..'))

# Import the specific functions needed to load exams, students, rooms, and timeslots
# Also import the tool that calculates which exams cannot happen at the same time
from utils.data_loader import (
    load_exams,
    load_students,
    load_rooms,
    load_timeslots,
    build_conflict_matrix
)

## Step 1 — Load Exams

Reads `data/data.csv`. Each exam gets a course code, student count, and a list of student IDs.

In [2]:
# Load all exam data from the CSV file into a dictionary named 'exams'
exams = load_exams(path='data/data.csv')

# --- Quick check ---
# Display the total number of unique exams found in the dataset
print(f"Total exams loaded: {len(exams)}")
print()

# Pick a specific course (ACCT201) to verify the data loaded correctly
sample_course = 'ACCT201'
print(f"Example — {sample_course}:")

# Show the total count of students registered for this specific exam
print(f"  Students enrolled : {exams[sample_course]['num']}")

# Show a small preview (the first 5 student IDs) to check the formatting
print(f"  First 5 IDs       : {exams[sample_course]['students'][:5]}")

Loaded 235 exams.
Total exams loaded: 235

Example — ACCT201:
  Students enrolled : 205
  First 5 IDs       : ['211001268', '231001594', '231002485', '241000049', '241001583']


In [3]:
# Sort all exams from largest to smallest based on the number of enrolled students
sorted_exams = sorted(exams.items(), key=lambda x: x[1]['num'], reverse=True)

# Print a header for the list of the 10 largest exams
print("Top 10 largest exams (hardest to fit in a room):")
print(f"{'Course':<20} {'Students':>10}")
print("-" * 32)

# Loop through the first 10 exams in the sorted list and print their name and student count
for course, info in sorted_exams[:10]:
    print(f"{course:<20} {info['num']:>10}")

Top 10 largest exams (hardest to fit in a room):
Course                 Students
--------------------------------
ECE151                      472
MTH113                      418
CSCI315                     385
PHY112                      354
MTH112                      328
CSCI217                     306
CSCI208                     295
CSCI305                     294
CSC103                      288
CSCI419                     288


## Step 2 — Load Students

Reads `data/IDs.csv`. Each student gets an ID and a list of their enrolled courses.

In [4]:
# Load all student data (IDs and their registered courses) from the CSV file
students = load_students(path='data/IDs.csv')

# Print the total number of unique students found in the dataset
print(f"Total students loaded: {len(students)}")
print()

# Pick the very first student ID from the list to show as an example
sample_id = list(students.keys())[0]
print(f"Example — Student {sample_id}:")

# List all the courses that this specific student is enrolled in
print(f"  Enrolled in: {students[sample_id]}")

Loaded 4265 students.
Total students loaded: 4265

Example — Student 1510203:
  Enrolled in: ['MKTG302', 'MKTG303', 'MKTG405', 'MKTG417', 'MKTG470', 'SSCI102']


In [5]:
# Create a list showing how many exams each individual student is signed up for
course_counts = [len(v) for v in students.values()]

# Print a header for the distribution summary
print("How many exams per student:")

# Loop through every possible number of exams (from 1 up to the maximum found)
for n in range(1, max(course_counts) + 1):
    # Count how many students have exactly 'n' exams scheduled
    count = course_counts.count(n)
    
    # Create a visual bar for a text chart (one block represents roughly 20 students)
    bar   = '█' * (count // 20)
    
    # Print the specific exam count, the number of students, and the visual bar
    print(f"  {n} exams: {count:>5} students  {bar}")

How many exams per student:
  1 exams:    89 students  ████
  2 exams:   277 students  █████████████
  3 exams:   739 students  ████████████████████████████████████
  4 exams:   968 students  ████████████████████████████████████████████████
  5 exams:  1242 students  ██████████████████████████████████████████████████████████████
  6 exams:   872 students  ███████████████████████████████████████████
  7 exams:    78 students  ███


## Step 3 — Load Rooms

Reads `data/rooms.csv`.

> ⚠️ **Make sure you have filled in the `capacity` column in rooms.csv before running this cell.**
> Rooms with empty capacity are skipped and cannot be used.

In [6]:
# Load all room data (ID, building name, and seating capacity) from the CSV file
rooms = load_rooms(path='data/rooms.csv')

# Print the total number of rooms available for exams
print(f"\nUsable rooms: {len(rooms)}")
print()

# Print a formatted header for the rooms table
print(f"{'Room ID':<10} {'Building':<20} {'Capacity':>10}")
print("-" * 42)

# Loop through each room and print its ID, location, and how many students it can hold
for r in rooms:
    print(f"{r['room_id']:<10} {r['building']:<20} {r['capacity']:>10}")

Loaded 38 usable rooms.

Usable rooms: 38

Room ID    Building               Capacity
------------------------------------------
1          Tarek Khalil                 33
7          Tarek Khalil                 33
8          Tarek Khalil                 33
9          Tarek Khalil                 33
52         Tarek Khalil                 33
53         Tarek Khalil                 33
104        Tarek Khalil                 65
106        Tarek Khalil                 64
110        Tarek Khalil                 62
111        Tarek Khalil                 66
114        Tarek Khalil                 63
116        Tarek Khalil                 66
129        Tarek Khalil                 68
132        Tarek Khalil                 65
134        Tarek Khalil                 66
138        Tarek Khalil                 64
139        Tarek Khalil                 66
142        Tarek Khalil                 66
264        Tarek Khalil                 35
265        Tarek Khalil                 35
306        

In [7]:
# Find the largest seating capacity available among all rooms
max_capacity  = max(r['capacity'] for r in rooms) if rooms else 0

# Identify the exam with the highest number of students
largest_exam  = sorted_exams[0]

# Print the capacity of the biggest room and the size of the biggest exam
print(f"Largest room capacity  : {max_capacity} students")
print(f"Largest exam           : {largest_exam[0]} with {largest_exam[1]['num']} students")
print()

# Check if the largest exam exceeds the size of the largest room
if max_capacity < largest_exam[1]['num']:
    # Warning: The exam is too big for one room and must be split into multiple rooms
    print("⚠️  WARNING: The largest exam does not fit in any single room.")
    print("   This means some exams will need to be split across multiple rooms.")
    print("   The GA will handle this by assigning the same exam to multiple rooms.")
else:
    # Confirmation: Every exam can fit into at least one of the existing rooms
    print("✓ All exams fit in at least one room.")

Largest room capacity  : 69 students
Largest exam           : ECE151 with 472 students

⚠️  WARNING: The largest exam does not fit in any single room.
   This means some exams will need to be split across multiple rooms.
   The GA will handle this by assigning the same exam to multiple rooms.


## Step 4 — Generate Timeslots

Creates all valid exam slots: Saturday 31 May to Monday 16 June,
working days only (Friday off), four 2-hour blocks per day.

In [8]:
# Load the full list of available dates and times (slots) for the exams
timeslots = load_timeslots()

# Print a formatted header for a table to display the schedule options
print()
print(f"{'Slot ID':<10} {'Date':<14} {'Day':<12} {'Time'}")
print("-" * 50)

# Loop through each individual timeslot and print its ID, date, weekday, and time
for ts in timeslots:
    print(f"{ts['slot_id']:<10} {ts['date']:<14} {ts['day']:<12} {ts['time']}")

Generated 60 timeslots across 15 working days.

Slot ID    Date           Day          Time
--------------------------------------------------
0          2025-05-31     Saturday     08:00-10:00
1          2025-05-31     Saturday     10:00-12:00
2          2025-05-31     Saturday     12:00-14:00
3          2025-05-31     Saturday     14:00-16:00
4          2025-06-01     Sunday       08:00-10:00
5          2025-06-01     Sunday       10:00-12:00
6          2025-06-01     Sunday       12:00-14:00
7          2025-06-01     Sunday       14:00-16:00
8          2025-06-02     Monday       08:00-10:00
9          2025-06-02     Monday       10:00-12:00
10         2025-06-02     Monday       12:00-14:00
11         2025-06-02     Monday       14:00-16:00
12         2025-06-03     Tuesday      08:00-10:00
13         2025-06-03     Tuesday      10:00-12:00
14         2025-06-03     Tuesday      12:00-14:00
15         2025-06-03     Tuesday      14:00-16:00
16         2025-06-04     Wednesday    08

## Step 5 — Build the Conflict Matrix

This is the most important step. For every pair of exams, we check if they
share at least one student. If they do, they **cannot** be placed in the same timeslot.

This may take 1–2 minutes to compute. The result is saved to `outputs/conflict_matrix.pkl`
so you never need to recompute it.

In [9]:
# Create a folder named 'outputs' if it doesn't already exist to store results
os.makedirs('outputs', exist_ok=True)

# Generate the conflict matrix (the list of exams that share students and can't overlap)
# Then, save this matrix as a file so we don't have to recalculate it later
conflict_matrix = build_conflict_matrix(
    exams,
    save_path='outputs/conflict_matrix.pkl'
)

Building conflict matrix (this may take a minute)...
Checked 27495 exam pairs.
Found 2702 conflicting pairs (share at least one student).
Conflict matrix saved to outputs/conflict_matrix.pkl


In [10]:
# Create a dictionary that counts how many other exams each specific course clashes with (shared students)
conflict_counts = {course: len(conflicts) for course, conflicts in conflict_matrix.items()}

# Sort the exams from most conflicts to least conflicts
sorted_conflicts = sorted(conflict_counts.items(), key=lambda x: x[1], reverse=True)

# Print a header for the "most conflicted" exams table
print("Top 10 exams with the most student-sharing conflicts:")
print(f"{'Course':<20} {'Conflicts with N other exams':>30}")
print("-" * 52)

# Loop through the top 10 most conflicted exams and print their name and conflict count
for course, count in sorted_conflicts[:10]:
    print(f"{course:<20} {count:>30}")

Top 10 exams with the most student-sharing conflicts:
Course                 Conflicts with N other exams
----------------------------------------------------
HUMA001/101 (Room)                              123
SSCI102                                          83
HUMA102                                          81
NSCI102                                          79
SSCI201                                          76
MTH113                                           66
CSCI208                                          56
MTH216                                           56
HUMA001/101 (Computer Lab)                             53
CSCI217                                          49


## Step 6 — Summary

Final check before moving to Notebook 2.

In [11]:
# Print a visual separator for the final data summary
print("=" * 50)
print("DATA PREPARATION SUMMARY")
print("=" * 50)

# Display the total number of exams that need to be scheduled
print(f"  Exams to schedule    : {len(exams)}")

# Display the total number of unique students involved
print(f"  Total students       : {len(students)}")

# Display the total number of rooms available for use
print(f"  Usable rooms         : {len(rooms)}")

# Display the total number of time slots created (date + time combinations)
print(f"  Available timeslots  : {len(timeslots)}")

# Calculate and display how many actual days the exam period lasts (assuming 4 slots per day)
print(f"  Working days         : {len(timeslots) // 4}")

# Calculate the total number of exam pairings that cannot happen at the same time
print(f"  Conflict pairs found : {sum(len(v) for v in conflict_matrix.values()) // 2}")

print("=" * 50)
print()

# Confirmation message that the data is processed and saved
print("✓ Data ready. Proceed to 02_constraints.ipynb")

DATA PREPARATION SUMMARY
  Exams to schedule    : 235
  Total students       : 4265
  Usable rooms         : 38
  Available timeslots  : 60
  Working days         : 15
  Conflict pairs found : 2702

✓ Data ready. Proceed to 02_constraints.ipynb
